# Trajectories of magnetic soft robot

In [44]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, Flow, FlowBody, Field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Parameters

In [45]:
EPSILON = 1E-6          # Small number

BLUEACCENT="#8FC2CC" 
REDACCENT="#C04D27"
GRAYFILL="#D6D6D6" 

In [46]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:      
  - spring_k    
  - radius
  - length
  - moment
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  spring_k: 10            
  radius0: 1
  radius1: 0.5
  length: 0.5
  moment: 3

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: radius0                    
    position: [0, 0, 0]            
    orientation: [0, 0, dof]
    torque: [0 * magnetic2, 0, -spring_k * dof - moment * magnetic1 * cos(dof) + moment * magnetic0 * sin(dof)]              

  - # Sphere 1 #################
    radius: radius1               
    position: [radius0 + radius1 + length, 0, 0]       
    orientation: [0, 0, 0] 
    torque: [0, 0, spring_k * dof]
"""

## Soft Plankton

In [47]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: dof, length, moment, radius0, radius1, spring_k, magnetic0, magnetic1, magnetic2
    Sphere 0
      Radius: radius0
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'dof']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '-dof*spring_k + magnetic0*moment*sin(dof) - magnetic1*moment*cos(dof)']
    Sphere 1
      Radius: radius1
      Position: ['length + radius0 + radius1', '0', '0']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', 'dof*spring_k']


In [48]:
tensors = mybody.compute_mobility_problem()
print("\nMobility Matrix M_H (multiplied by mu):")
print(tensors.M_H)
print("\nMobility Matrix M_K (multiplied by mu):")
print(tensors.M_K)


Mobility Matrix M_H (multiplied by mu):
[[ 0.     0.     0.   ]
 [ 0.    -0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.    -0.013  0.   ]
 [ 0.    -0.106  0.   ]]

Mobility Matrix M_K (multiplied by mu):
[[ 0.   ]
 [-0.153]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.227]
 [-0.579]]


In [49]:
class NoFlow(Flow):
    """Quiet fluid."""
    def _velocity(self, pos):
        return jnp.array([0, 0, 0])
    
class OscillatingMagneticField(Field):
    """Oscillating magnetic field (constant component along x, oscillating component along y)."""

    def _field_vector(self, pos, time):
        B0x = self.params[0] if self.params else 1.0  # Constant component of the field
        B0y = self.params[1] if self.params else 1.0  # Oscillating component of the field
        omega = self.params[2] if self.params else 1.0  # Default angular velocity
        return jnp.array([B0x, B0y * jnp.cos(omega * time), 0])

In [50]:
# Create a shear flow with shear rate 1
no_flow = NoFlow()
# Create a gravity field of acceleration 0
magnetic_field = OscillatingMagneticField(1., 1.5, 1.)

# Test it (steady case)
pos = [1.0, 2.0, 3.0] 
time= 1 
grad_u = no_flow.gradient(pos)
Omega, rate_of_strain = no_flow.omega_s(pos)  
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain E):\n", rate_of_strain)
print("Magnetic field", magnetic_field.vector(pos, time))


Velocity Gradient Tensor (∇u):
 [[ 0  0  0]
 [ 0  0  0]
 [ 0  0  0]]
Angular velocity Ω: [ 0.  0.  0.]
Rate-of-strain E):
 [[ 0.  0.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Magnetic field [ 1.    0.81  0.  ]


In [51]:
from softmobility.classes.flowbody import rotation_matrix_from_Rodrigues

orientation = [0, 0,1]
R, sixc_R = rotation_matrix_from_Rodrigues(orientation, Ndof=mybody.Ndof)

print(R)
print(sixc_R)


[[ 0.54  -0.841  0.   ]
 [ 0.841  0.54   0.   ]
 [ 0.     0.     1.   ]]
[[ 0.54  -0.841  0.     0.     0.     0.     0.   ]
 [ 0.841  0.54   0.     0.     0.     0.     0.   ]
 [ 0.     0.     1.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.54  -0.841  0.     0.   ]
 [ 0.     0.     0.     0.841  0.54   0.     0.   ]
 [ 0.     0.     0.     0.     0.     1.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     1.   ]]


In [86]:
# parameters
final_time = 6 * jnp.pi  
dt = 0.1

mybody.set_design_defaults(new_dict={'spring_k': 20, 'radius1': 0.5, 'length': 0., 'moment': 3})


solver = FlowBody(mybody, no_flow, magnetic_field, dt=dt, init_orientation=[0, 0, 0], integrator="RK2")
trajectory = solver.simulate(T=final_time)

OLD default param values: ['length', 'moment', 'radius0', 'radius1', 'spring_k'] [  0.    3.    1.    0.5  20. ]
NEW default param values: ['length', 'moment', 'radius0', 'radius1', 'spring_k'] [  0.    3.    1.    0.5  20. ]
Time: 0.000 / 18.850  Integrator RK2
Time: 10.000 / 18.850  Integrator RK2


In [87]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs, mode='lines', name='DOF'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()